# 08 — Build the Portfolio Semantic Layer

In Notebook 07, we built the **Research Agent** over unstructured research.

In this notebook, we build the governed **structured portfolio semantic layer** that the future Portfolio Agent will use.

### Goal

Create a reusable portfolio layer that can answer questions such as:

- How much do we own?
- What is the market value by company or portfolio?
- What is our sector exposure?
- What is the cost basis and unrealized gain/loss?
- What did we own as of a specific date?

### Architecture

```text
Existing Portfolio Delta Tables
            ↓
Canonical Portfolio Semantic Source
            ↓
Unity Catalog Metric View
            ↓
Business Metrics + Agent Metadata
            ↓
Future Genie / Portfolio Agent
```

We are **not building the Portfolio Agent yet**.  
First, we create the governed business semantics that the agent will depend on.

## 1. Configure the portfolio semantic layer

We keep the main object names in one place.

The notebook will:

1. inspect the existing `investment_ai.portfolio` schema,
2. identify the portfolio-position source,
3. standardize it into one canonical semantic source,
4. build a Metric View over that source.

In [0]:
CATALOG = "investment_ai"
SCHEMA = "portfolio"

SEMANTIC_SOURCE_VIEW = f"{CATALOG}.{SCHEMA}.portfolio_semantic_source"
METRIC_VIEW = f"{CATALOG}.{SCHEMA}.portfolio_metrics"

# Fixed business-analysis date for this POC.
POC_AS_OF_DATE = "2026-06-30"

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Semantic source:", SEMANTIC_SOURCE_VIEW)
print("Metric view:", METRIC_VIEW)

## 2. Inspect the existing portfolio objects

Before creating any semantic model, we verify what already exists.

This keeps Notebook 08 connected to the portfolio data created earlier in the POC rather than creating a second copy of the business data.

In [0]:
portfolio_objects = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
display(portfolio_objects)

## 3. Discover the portfolio source and column mappings

The POC has evolved over time, so this cell makes the semantic layer tolerant of common portfolio table and column names.

It looks for a position/holdings table first, and then optional company, fund, and price reference tables.

The code does **not** silently create portfolio data.  
If the required portfolio fields cannot be found, it stops and tells us what is missing.

In [0]:
tables = {
    row.tableName.lower(): row.tableName
    for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
}

def first_existing(candidates):
    for candidate in candidates:
        if candidate.lower() in tables:
            return tables[candidate.lower()]
    return None

def table_columns(table_name):
    if not table_name:
        return {}
    rows = spark.sql(
        f"DESCRIBE TABLE {CATALOG}.{SCHEMA}.`{table_name}`"
    ).collect()
    return {
        row.col_name.lower(): row.col_name
        for row in rows
        if row.col_name and not row.col_name.startswith("#")
    }

def find_column(columns, aliases, required=False, label=None):
    for alias in aliases:
        if alias.lower() in columns:
            return columns[alias.lower()]
    if required:
        raise ValueError(
            f"Could not find required field '{label or aliases[0]}'. "
            f"Tried aliases: {aliases}. Available columns: {sorted(columns.values())}"
        )
    return None

POSITION_TABLE = first_existing([
    "current_positions",
    "portfolio_state",
    "holdings",
    "positions",
    "fact_positions_daily",
    "position",
])

COMPANY_TABLE = first_existing([
    "companies",
    "security_master",
    "issuer_master",
])

FUND_TABLE = first_existing([
    "funds",
    "portfolio_master",
])

PRICE_TABLE = first_existing([
    "prices",
    "price",
])

if POSITION_TABLE is None:
    raise ValueError(
        f"No recognized portfolio position table was found in {CATALOG}.{SCHEMA}. "
        "Expected one of: current_positions, portfolio_state, holdings, positions, "
        "fact_positions_daily, position."
    )

position_cols = table_columns(POSITION_TABLE)
company_cols = table_columns(COMPANY_TABLE)
fund_cols = table_columns(FUND_TABLE)
price_cols = table_columns(PRICE_TABLE)

mapping = {
    "position_table": POSITION_TABLE,
    "company_table": COMPANY_TABLE,
    "fund_table": FUND_TABLE,
    "price_table": PRICE_TABLE,

    "position_security_id": find_column(
        position_cols,
        ["security_id", "symbol", "ticker"],
        required=True,
        label="security_id",
    ),
    "position_portfolio_id": find_column(
        position_cols,
        ["portfolio_id", "fund_id", "account_id"],
        required=True,
        label="portfolio_id / fund_id",
    ),
    "position_as_of_date": find_column(
        position_cols,
        ["as_of_date", "snapshot_date", "position_date", "business_date", "date"],
        required=True,
        label="as_of_date",
    ),
    "quantity": find_column(
        position_cols,
        ["quantity", "position_qty", "shares", "shares_held"],
    ),
    "market_value": find_column(
        position_cols,
        ["market_value", "market_value_usd", "position_value", "holding_value"],
    ),
    "market_price": find_column(
        position_cols,
        ["market_price", "price", "close_price", "last_price"],
    ),
    "cost_basis": find_column(
        position_cols,
        ["cost_basis", "cost_basis_usd", "book_value", "cost_value"],
    ),
    "avg_cost": find_column(
        position_cols,
        ["avg_cost", "average_cost", "avg_price", "average_price"],
    ),
    "company_name_direct": find_column(
        position_cols,
        ["company_name", "issuer_name", "security_name"],
    ),
    "portfolio_name_direct": find_column(
        position_cols,
        ["portfolio_name", "fund_name", "account_name"],
    ),
    "sector_direct": find_column(
        position_cols,
        ["sector", "gics_sector"],
    ),
}

print("Detected portfolio source:")
for key, value in mapping.items():
    print(f"  {key}: {value}")

## 4. Build one canonical portfolio semantic source

Agents should not need to understand every physical source-table variation.

We standardize the portfolio data into one governed view with consistent business columns:

- `as_of_date`
- `portfolio_id`
- `portfolio_name`
- `security_id`
- `company_name`
- `sector`
- `quantity`
- `market_price`
- `market_value`
- `cost_basis`
- `unrealized_gain_loss`
- `position_weight`

If a descriptive field is not present in the position table, the notebook attempts to enrich it from the existing company, fund, or price tables.

In [0]:
def q(name):
    return f"`{name}`"

pos = f"{CATALOG}.{SCHEMA}.`{POSITION_TABLE}`"

# ---------- Optional company enrichment ----------
company_join = ""
company_name_expr = "CAST(NULL AS STRING)"
sector_expr = "CAST(NULL AS STRING)"

if mapping["company_name_direct"]:
    company_name_expr = f"p.{q(mapping['company_name_direct'])}"
if mapping["sector_direct"]:
    sector_expr = f"p.{q(mapping['sector_direct'])}"

if COMPANY_TABLE and (not mapping["company_name_direct"] or not mapping["sector_direct"]):
    company_security = find_column(
        company_cols, ["security_id", "symbol", "ticker"], required=False
    )
    company_name = find_column(
        company_cols, ["company_name", "issuer_name", "security_name", "name"], required=False
    )
    company_sector = find_column(
        company_cols, ["sector", "gics_sector"], required=False
    )

    if company_security:
        company_join = f"""
        LEFT JOIN {CATALOG}.{SCHEMA}.`{COMPANY_TABLE}` c
          ON p.{q(mapping['position_security_id'])} = c.{q(company_security)}
        """
        if not mapping["company_name_direct"] and company_name:
            company_name_expr = f"c.{q(company_name)}"
        if not mapping["sector_direct"] and company_sector:
            sector_expr = f"c.{q(company_sector)}"

# ---------- Optional fund enrichment ----------
fund_join = ""
portfolio_name_expr = "CAST(NULL AS STRING)"

if mapping["portfolio_name_direct"]:
    portfolio_name_expr = f"p.{q(mapping['portfolio_name_direct'])}"
elif FUND_TABLE:
    fund_id = find_column(
        fund_cols, ["fund_id", "portfolio_id", "account_id"], required=False
    )
    fund_name = find_column(
        fund_cols, ["fund_name", "portfolio_name", "account_name", "name"], required=False
    )
    if fund_id:
        fund_join = f"""
        LEFT JOIN {CATALOG}.{SCHEMA}.`{FUND_TABLE}` f
          ON p.{q(mapping['position_portfolio_id'])} = f.{q(fund_id)}
        """
        if fund_name:
            portfolio_name_expr = f"f.{q(fund_name)}"

# ---------- Optional price enrichment ----------
price_join = ""
market_price_expr = (
    f"CAST(p.{q(mapping['market_price'])} AS DOUBLE)"
    if mapping["market_price"]
    else "CAST(NULL AS DOUBLE)"
)

if PRICE_TABLE and not mapping["market_price"]:
    price_security = find_column(
        price_cols, ["security_id", "symbol", "ticker"], required=False
    )
    price_date = find_column(
        price_cols, ["as_of_date", "price_date", "date", "business_date"], required=False
    )
    price_value = find_column(
        price_cols, ["market_price", "price", "close_price", "last_price"], required=False
    )

    if price_security and price_value:
        date_condition = ""
        if price_date:
            date_condition = (
                f" AND p.{q(mapping['position_as_of_date'])} = pr.{q(price_date)}"
            )

        price_join = f"""
        LEFT JOIN {CATALOG}.{SCHEMA}.`{PRICE_TABLE}` pr
          ON p.{q(mapping['position_security_id'])} = pr.{q(price_security)}
          {date_condition}
        """
        market_price_expr = f"CAST(pr.{q(price_value)} AS DOUBLE)"

quantity_expr = (
    f"CAST(p.{q(mapping['quantity'])} AS DOUBLE)"
    if mapping["quantity"]
    else "CAST(NULL AS DOUBLE)"
)

if mapping["market_value"]:
    market_value_expr = f"CAST(p.{q(mapping['market_value'])} AS DOUBLE)"
elif mapping["quantity"] and market_price_expr != "CAST(NULL AS DOUBLE)":
    market_value_expr = f"({quantity_expr} * {market_price_expr})"
else:
    raise ValueError(
        "The semantic layer needs either an existing market_value field, "
        "or both quantity and market price so market value can be calculated."
    )

if mapping["cost_basis"]:
    cost_basis_expr = f"CAST(p.{q(mapping['cost_basis'])} AS DOUBLE)"
elif mapping["quantity"] and mapping["avg_cost"]:
    cost_basis_expr = (
        f"({quantity_expr} * CAST(p.{q(mapping['avg_cost'])} AS DOUBLE))"
    )
else:
    cost_basis_expr = "CAST(NULL AS DOUBLE)"

semantic_source_sql = f"""
CREATE OR REPLACE VIEW {SEMANTIC_SOURCE_VIEW} AS
WITH standardized AS (
    SELECT
        CAST(p.{q(mapping['position_as_of_date'])} AS DATE) AS as_of_date,
        CAST(p.{q(mapping['position_portfolio_id'])} AS STRING) AS portfolio_id,
        CAST({portfolio_name_expr} AS STRING) AS portfolio_name,
        CAST(p.{q(mapping['position_security_id'])} AS STRING) AS security_id,
        CAST({company_name_expr} AS STRING) AS company_name,
        CAST({sector_expr} AS STRING) AS sector,
        {quantity_expr} AS quantity,
        {market_price_expr} AS market_price,
        {market_value_expr} AS market_value,
        {cost_basis_expr} AS cost_basis
    FROM {pos} p
    {company_join}
    {fund_join}
    {price_join}
),
calculated AS (
    SELECT
        *,
        CASE
            WHEN cost_basis IS NULL THEN NULL
            ELSE market_value - cost_basis
        END AS unrealized_gain_loss,
        CASE
            WHEN SUM(market_value) OVER (
                PARTITION BY as_of_date, portfolio_id
            ) = 0 THEN NULL
            ELSE market_value / SUM(market_value) OVER (
                PARTITION BY as_of_date, portfolio_id
            )
        END AS position_weight
    FROM standardized
)
SELECT * FROM calculated
"""

spark.sql(semantic_source_sql)

print(f"✅ Created semantic source: {SEMANTIC_SOURCE_VIEW}")

## 5. Validate the canonical portfolio source

Before defining business metrics, we verify the standardized output.

For our POC, the key business date is **June 30, 2026**.

In [0]:
display(
    spark.sql(
        f'''
        SELECT *
        FROM {SEMANTIC_SOURCE_VIEW}
        WHERE as_of_date = DATE '{POC_AS_OF_DATE}'
        ORDER BY portfolio_id, market_value DESC
        '''
    )
)

## 6. Create the Unity Catalog Metric View

The Metric View is our governed semantic contract.

Instead of asking every downstream application to redefine metrics such as market value or unrealized gain/loss, we define those business measures once.

We also add **agent metadata**:

- display names,
- business descriptions,
- formatting,
- synonyms.

This makes the same semantic layer easier for SQL users, dashboards, and future Genie Agents to understand.

In [0]:
metric_view_yaml = f"""
version: 1.1
comment: >
  Governed portfolio semantic layer for Investment AI.
  Provides reusable ownership, valuation, exposure, and unrealized P&L metrics.

source: {SEMANTIC_SOURCE_VIEW}

fields:
  - name: as_of_date
    expr: as_of_date
    comment: Portfolio position snapshot date.
    display_name: As Of Date
    format:
      type: date
      date_format: year_month_day
      leading_zeros: true
    synonyms:
      - position date
      - snapshot date
      - holdings date

  - name: portfolio_id
    expr: portfolio_id
    comment: Unique identifier for the investment portfolio or fund.
    display_name: Portfolio ID
    synonyms:
      - fund id
      - account id

  - name: portfolio_name
    expr: portfolio_name
    comment: Business-friendly portfolio or fund name.
    display_name: Portfolio Name
    synonyms:
      - fund
      - fund name
      - portfolio

  - name: security_id
    expr: security_id
    comment: Unique identifier of the held security or issuer.
    display_name: Security ID
    synonyms:
      - security
      - ticker
      - symbol

  - name: company_name
    expr: company_name
    comment: Company or issuer associated with the held security.
    display_name: Company Name
    synonyms:
      - company
      - issuer
      - issuer name

  - name: sector
    expr: sector
    comment: Business sector associated with the held security.
    display_name: Sector
    synonyms:
      - industry sector
      - sector exposure

  - name: position_weight
    expr: position_weight
    comment: Position market value divided by total market value of the same portfolio on the same date.
    display_name: Position Weight
    format:
      type: percentage
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
    synonyms:
      - portfolio weight
      - holding weight
      - exposure weight

measures:
  - name: total_market_value
    expr: SUM(market_value)
    comment: Total market value of selected portfolio positions in US dollars.
    display_name: Total Market Value
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
      abbreviation: compact
    synonyms:
      - market value
      - holdings value
      - portfolio value
      - exposure

  - name: total_quantity
    expr: SUM(quantity)
    comment: Total position quantity or shares held.
    display_name: Total Quantity
    format:
      type: number
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
    synonyms:
      - shares
      - shares owned
      - units
      - quantity owned

  - name: total_cost_basis
    expr: SUM(cost_basis)
    comment: Total cost basis of selected positions when cost information is available.
    display_name: Total Cost Basis
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
      abbreviation: compact
    synonyms:
      - cost basis
      - book cost
      - invested cost

  - name: unrealized_gain_loss
    expr: SUM(unrealized_gain_loss)
    comment: Aggregate market value minus aggregate cost basis for selected positions.
    display_name: Unrealized Gain Loss
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
      abbreviation: compact
    synonyms:
      - unrealized pnl
      - unrealized p and l
      - unrealized profit
      - unrealized loss

  - name: security_count
    expr: COUNT(DISTINCT security_id)
    comment: Number of distinct securities in the selected portfolio slice.
    display_name: Security Count
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
      hide_group_separator: false
    synonyms:
      - number of securities
      - number of holdings
      - holdings count

  - name: average_position_value
    expr: AVG(market_value)
    comment: Average market value of positions in the selected portfolio slice.
    display_name: Average Position Value
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: max
        places: 2
      hide_group_separator: false
      abbreviation: compact
    synonyms:
      - average holding value
      - average position size
"""

create_metric_view_sql = f"""
CREATE OR REPLACE VIEW {METRIC_VIEW}
WITH METRICS
LANGUAGE YAML
AS
$$
{metric_view_yaml}
$$
"""

spark.sql(create_metric_view_sql)

print(f"✅ Created Metric View: {METRIC_VIEW}")

## 7. Test the governed metrics

Metric View measures are evaluated with the `MEASURE()` function.

First, we test total market value by portfolio for the POC snapshot date.

In [0]:
display(
    spark.sql(
        f'''
        SELECT
            portfolio_name,
            MEASURE(total_market_value) AS total_market_value,
            MEASURE(security_count) AS security_count
        FROM {METRIC_VIEW}
        WHERE as_of_date = DATE '{POC_AS_OF_DATE}'
        GROUP BY portfolio_name
        ORDER BY total_market_value DESC
        '''
    )
)

## 8. Test company exposure

This is the semantic pattern the future Portfolio Agent will use when a user asks:

> "How much do we own of this company?"

The agent will supply the company/security filter and the Metric View will provide the governed measure.

In [0]:
display(
    spark.sql(
        f'''
        SELECT
            security_id,
            company_name,
            MEASURE(total_market_value) AS total_market_value,
            MEASURE(total_quantity) AS total_quantity
        FROM {METRIC_VIEW}
        WHERE as_of_date = DATE '{POC_AS_OF_DATE}'
          AND security_id = 'IA-SYN-01'
        GROUP BY security_id, company_name
        '''
    )
)

## 9. Test sector exposure

The same Metric View can answer portfolio-level, company-level, and sector-level questions without redefining the underlying measures.

In [0]:
display(
    spark.sql(
        f'''
        SELECT
            sector,
            MEASURE(total_market_value) AS total_market_value
        FROM {METRIC_VIEW}
        WHERE as_of_date = DATE '{POC_AS_OF_DATE}'
        GROUP BY sector
        ORDER BY total_market_value DESC
        '''
    )
)

## 10. Validate the business question boundary

Our end-to-end POC question is:

> **Which companies lowered their outlook in Q2 2026, and how much do we own as of June 30, 2026?**

The responsibilities are intentionally separated:

```text
Research Agent
    ↓
Identifies the companies that lowered outlook
    ↓
Returns security/company identifiers
    ↓
Portfolio Agent (future)
    ↓
Queries portfolio_metrics
    ↓
Returns quantity / market value / exposure
```

Notebook 08 prepares the **structured portfolio side** of that workflow.

The Research Agent should not calculate portfolio ownership, and the Portfolio semantic layer should not attempt to interpret research documents.

# ✅ Notebook 08 complete

We now have:

- a canonical portfolio semantic source,
- governed portfolio business metrics,
- agent-friendly display names and synonyms,
- consistent currency and percentage formatting,
- date-aware portfolio ownership,
- a semantic layer ready for Genie.

### Next notebook

**09 — Build the Genie Portfolio Agent**

In Notebook 09, we will connect Genie to this Metric View and teach it to answer structured portfolio questions in natural language.